##  Step 1: Install Dependencies

In [1]:
!pip install -q \
  torch \
  transformers \
  datasets \
  peft \
  bitsandbytes \
  accelerate \
  trl


In [2]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "train.jsonl",
        "validation": "val.jsonl"
    }
)

dataset


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 1080
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 120
    })
})

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)
tokenizer.pad_token = tokenizer.eos_token


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
MAX_LENGTH = 512

def format_instruction(example):
    if example["input"].strip() != "":
        prompt = f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    else:
        prompt = f"""### Instruction:
{example['instruction']}

### Response:
{example['output']}"""

    return tokenizer(
        prompt,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )


In [5]:
tokenized_ds = dataset.map(
    format_instruction,
    remove_columns=dataset["train"].column_names,
    batched=False
)


Map:   0%|          | 0/1080 [00:00<?, ? examples/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

In [6]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model.config.use_cache = False


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [7]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [8]:
model.gradient_checkpointing_enable()


In [11]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=500,
    eval_strategy="steps",
    save_strategy="epoch",
    eval_steps=500,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)


In [12]:
from transformers import Trainer, DataCollatorForLanguageModeling

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )
)

trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
500,1.155689,1.283883


TrainOutput(global_step=810, training_loss=1.3404041878971054, metrics={'train_runtime': 970.1863, 'train_samples_per_second': 3.34, 'train_steps_per_second': 0.835, 'total_flos': 1.031921417060352e+16, 'train_loss': 1.3404041878971054, 'epoch': 3.0})

## 🎓 Step 11: Train the Model

In [13]:
adapter_path = "./adapters"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

('./adapters/tokenizer_config.json',
 './adapters/chat_template.jinja',
 './adapters/tokenizer.json')

In [20]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=bnb_config
)

model = PeftModel.from_pretrained(base_model, adapter_path)
prompt = """### Instruction:
Answer the medical question accurately.

### Input:
What is the outlook for Lip and Oral Cavity Cancer ?
### Response:
"""
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.6,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
response = decoded.split("### Response:")[-1].strip()

print("\n" + "=" * 80)
print("MODEL RESPONSE")
print("=" * 80)
print(response)
print("=" * 80)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


MODEL RESPONSE
Certain factors affect prognosis (chance of recovery) and treatment options. The prognosis (chance of recovery) and treatment options depend on the following:         - Whether the cancer is early stage or advanced.    - Whether the tumor has metastasized to other parts of the body, including bones and lymph nodes.    - Whether the cancer is located in a region where the risk of recurrence is high.        Treatment options may include the following:     - Radiation therapy.    - Chemotherapy.    - Hormonal therapy.    - Surgery.    - Immunotherapy.    - Targeted therapy.    - Radiosensitization
